
# 🚀 Core Vision Perfect V1 — 01b · Caption BOOST (chạy SONG SONG với nb01)

Notebook phụ **chỉ chạy captions**, đi **NGƯỢC** danh sách video (L30 → L21)
trong khi phiên nb01 chính đi xuôi (L21 → L30) — hai phiên tự **gặp nhau ở
giữa** nhờ resume-skip theo từng video, chia đôi thời gian captions.

**An toàn:** mỗi video là MỘT file json ghi atomic trên Drive; phiên này KHÔNG
đụng embeddings / FAISS / BM25 / OCR / ASR. Tệ nhất hai phiên trùng nhau đúng
1 video ở điểm gặp — bên sau ghi đè bản y hệt, không thể phá artifact.

Cách dùng: mở **một phiên Colab GPU thứ hai** (A100/H100 đều được) → Run all.
Đứt phiên → Run all lại, tự resume. Khi CẢ HAI phiên xong captions: chạy nb01
một lượt cuối với `FORCE_TEXT_INDEX=True` để BM25 nạp đủ trường caption.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 · PARAMS — the ONLY cell you may need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════╝
DRIVE_PROJECT_DIR = "AIC2025"        # MyDrive/<this>/{data, artifacts}
REPO_URL  = "https://github.com/ledinhminhquan/Core-Vision_Perfect_V1.git"
REPO_REF  = "main"

# Which dense encoders to build indexes for (order = ensemble order).
#   "siglip2"          multilingual default (needed for training too)
#   "openclip"         English lane (DFN5B ViT-H/14-378) — strongest with translation
#   "qwen_embed"       optional HEAVY lane (Qwen embedding tower — strong, slow)
#   "provided_clip32"  organiser features — instant, no GPU (L-batches only);
#                      auto-added in the catalog cell when clip-features-32 exists
EMBED_MODELS = ["siglip2", "openclip"]

# Copy keyframes from Drive → local disk before embedding (much faster I/O).
COPY_KEYFRAMES_LOCAL = True

# Aux indexes to build (each is resumable; captions are the slowest).
RUN_OCR, RUN_ASR, RUN_CAPTIONS = True, True, True
CAPTION_STRIDE = 4                   # caption mỗi keyframe thứ 4 (round-16: đủ dày
#   cho kênh recall BM25 mà nhanh gấp đôi stride 2. NÂNG stride luôn an toàn với
#   resume: video đã caption ở stride nhỏ hơn vẫn được tính là XONG ở stride lớn
#   hơn. Mọi phiên chạy song song PHẢI dùng CÙNG một stride.

# K-batch shot detection: install TransNetV2 (the winning-team detector) for
# keyframe self-extraction. Installed --no-deps (Colab torch is never touched);
# without it extraction falls back to PySceneDetect automatically. (nb01 only)
INSTALL_TRANSNETV2 = True

# Force-rebuild toggles — mặc định False = resume/skip khi artifact đã có.
FORCE_CATALOG    = False             # rebuild the catalog parquet
FORCE_EMBED      = False             # re-embed every keyframe
FORCE_INDEX      = False             # rebuild the FAISS indexes
FORCE_AUX        = False             # redo OCR/ASR/captions from scratch
FORCE_TEXT_INDEX = False             # rebuild the persisted BM25 text index

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("params ok")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 · Mount Drive + folder layout + preflight write test          ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess, time
from pathlib import Path

from google.colab import drive

_MP = "/content/drive"

def _drive_alive() -> bool:
    try:
        return Path(_MP, "MyDrive").exists()
    except OSError:
        return False

def _ensure_drive():
    """Mount / HỒI SINH Drive FUSE — dùng ở mọi cell dài hơi phía sau.

    Round-19 (live run 8): daemon DriveFS chết để lại mountpoint 'bẩn' →
    drive.mount kêu 'Mountpoint must not already contain files' và cả
    force_remount cũng bó tay. Trình tự cứu đúng: (1) fusermount -uz gỡ
    mount chết; (2) CHỈ khi chắc chắn không còn mount (os.path.ismount ==
    False — lúc này các entry trong mountpoint là RÁC LOCAL trên đĩa VM,
    không phải Drive thật) mới dọn sạch chúng; (3) mount lại.
    """
    for _try in range(4):
        if _drive_alive():
            return
        if _try:
            print(f"⚠ Drive FUSE chưa sống — hồi sinh (lần {_try}/3) ...")
        try:
            if os.path.ismount(_MP):
                subprocess.run(["fusermount", "-uz", _MP], capture_output=True)
                time.sleep(2)
            if os.path.isdir(_MP) and not os.path.ismount(_MP):
                for _c in os.listdir(_MP):     # rác local — KHÔNG phải Drive
                    _p = os.path.join(_MP, _c)
                    shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) \
                        else os.unlink(_p)
            drive.mount(_MP, force_remount=bool(_try))
        except Exception as _e:  # noqa: BLE001 — thử tiếp vòng sau
            print("   mount lỗi:", _e)
            time.sleep(5)
    if not _drive_alive():
        raise RuntimeError(
            "Không mount được Google Drive sau 4 lần thử — Runtime ▸ "
            "Disconnect and delete runtime rồi Run all lại (tiến độ đã lưu "
            "trên Drive còn nguyên).")

_ensure_drive()
assert Path("/content/drive/MyDrive").exists(), "Drive mount failed — rerun this cell"

PROJECT   = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
DATA_DIR  = PROJECT / "data"           # organiser dataset (merged packages)
ARTIFACTS = PROJECT / "artifacts"      # everything we build → survives disconnects
for p in (DATA_DIR, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)

# PREFLIGHT (v12): Drive PHẢI ghi/đọc được — quota đầy hay mất quyền thì
# dừng NGAY tại đây thay vì hỏng giữa chừng sau 2 giờ chạy.
_probe = ARTIFACTS / f"_write_test_{int(time.time())}.tmp"
try:
    _probe.write_text("ok", encoding="utf-8")
    assert _probe.read_text(encoding="utf-8") == "ok"
    _probe.unlink()
    print("✅ Drive write test: OK")
except Exception as e:
    raise RuntimeError(
        f"❌ Không ghi được vào Drive ({ARTIFACTS}): {e!r}\n"
        "Kiểm tra dung lượng (quota) Google Drive và quyền truy cập thư mục, "
        "rồi chạy lại ô này."
    ) from e

# DATA-PRESENCE GATE (round-20, live run 9): trên VM mới, DriveFS có thể liệt
# kê data/ ra RỖNG suốt vài phút đầu (metadata sync lười) — mkdir exist_ok ở
# trên còn CHE mất triệu chứng, để cell 7 chết khó hiểu với "Keyframes folder
# not found". Poll tới 3 phút (mỗi listdir là một cú hích ép DriveFS fetch);
# hết kiên nhẫn thì dừng TO với chẩn đoán rõ ràng.
_t0 = time.time()
_data_ok = False
while time.time() - _t0 < 180:
    try:
        if any(DATA_DIR.iterdir()):
            _data_ok = True
            break
    except OSError:
        pass
    print(f"⏳ data/ đang rỗng — đợi DriveFS sync metadata ({int(time.time() - _t0)}s) ...")
    time.sleep(10)
if not _data_ok:
    raise RuntimeError(
        "data/ trên Drive vẫn RỖNG sau 3 phút chờ. Ba nguyên nhân thường gặp:\n"
        "  1) Phiên Colab đăng nhập NHẦM tài khoản Google (kiểm tra avatar góc "
        f"phải trên) — phải là tài khoản có MyDrive/{DRIVE_PROJECT_DIR}/data;\n"
        "  2) DriveFS sync quá chậm — Runtime ▸ Disconnect and delete runtime "
        "rồi Run all lại trên máy mới;\n"
        "  3) Lần chạy đầu tiên mà chưa upload dữ liệu — ném các zip của BTC "
        f"vào MyDrive/{DRIVE_PROJECT_DIR}/data trước (docs/DRIVE_SETUP.md).\n"
        "KHÔNG có gì bị mất — dữ liệu vẫn nằm nguyên trên Drive của tài khoản đúng.")
print(f"✅ data/ nhìn thấy dữ liệu sau {int(time.time() - _t0)}s")

# HF + pip caches on Drive → models/wheels download once, not per session.
os.environ["HF_HOME"] = str(ARTIFACTS / "hf_cache")
os.environ["PIP_CACHE_DIR"] = str(ARTIFACTS / "pip_cache")
for _d in (os.environ["HF_HOME"], os.environ["PIP_CACHE_DIR"]):
    Path(_d).mkdir(parents=True, exist_ok=True)

import shutil
free_gb = shutil.disk_usage(str(PROJECT)).free / 1e9
print(f"Project: {PROJECT}")
print(f"Drive free space: {free_gb:.0f} GB")
if free_gb < 20:
    print("⚠ Less than 20 GB free on Drive — embeddings/checkpoints may not fit!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 · Get repo + install dependencies (v12 discipline)            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Quy tắc (học từ notebook Toxicity v12):
#   * check version qua importlib.metadata — KHÔNG import package trước khi
#     nâng cấp (import sớm sẽ ghim version cũ vào sys.modules);
#   * KHÔNG BAO GIỜ đụng torch/torchvision/torchaudio của Colab;
#   * chỉ cài đúng những gói thiếu/sai version (--prefer-binary);
#   * sau khi cài: `pip check` + micro-fix (tối đa 2 vòng, không crash),
#     rồi purge sys.modules TRƯỚC khi import cvp.
FORCE_REINSTALL_DEPS = False

import re, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Core-Vision_Perfect_V1")

def _run(cmd, show=None, **kw):
    # `show` masks credentials in the echoed command — a PAT-carrying clone
    # URL must NEVER be printed into the saved notebook output.
    print("$", " ".join(map(str, show or cmd)))
    return subprocess.run([str(c) for c in cmd], check=False, **kw).returncode

def _pip(args):
    return _run([sys.executable, "-m", "pip", *args])

# Private repo? Add a fine-grained PAT as Colab secret "GITHUB_TOKEN"
# (Contents: Read-only on this repo) and ENABLE its notebook-access toggle.
clone_url, _tok = REPO_URL, None
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as _e:
    print(f"⚠ KHÔNG đọc được secret GITHUB_TOKEN ({type(_e).__name__}) — repo "
          "private sẽ KHÔNG clone được. Kiểm tra: 🔑 panel có secret tên đúng "
          "y hệt GITHUB_TOKEN và công tắc 'Notebook access' đã BẬT chưa?")
if _tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{_tok}@")
    print(f"GITHUB_TOKEN: loaded ({len(_tok)} chars, {_tok[:11]}…)")
elif not _tok:
    print("⚠ GITHUB_TOKEN trống/vắng mặt — thử clone KHÔNG xác thực "
          "(chắc chắn fail nếu repo private).")

if REPO_DIR.exists():
    _run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])
    _run(["git", "-C", REPO_DIR, "checkout", REPO_REF, "-q"])
    _run(["git", "-C", REPO_DIR, "pull", "-q"])
else:
    rc = _run(["git", "clone", "--branch", REPO_REF, clone_url, REPO_DIR],
              show=["git", "clone", "--branch", REPO_REF, REPO_URL, REPO_DIR])
    if rc != 0:  # private repo / no network → fall back to a Drive copy
        print("⚠ Clone THẤT BẠI. Nguyên nhân thường gặp, theo thứ tự:\n"
              "  1) Secret GITHUB_TOKEN sai tên / chưa bật Notebook access "
              "(xem cảnh báo phía trên);\n"
              "  2) PAT sai/hết hạn/thiếu quyền — cần fine-grained PAT với "
              "Contents: Read-only cấp cho ĐÚNG repo này;\n"
              "  3) Mạng Colab trục trặc tạm thời — chạy lại cell.")
        drive_copy = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR / "Core-Vision_Perfect_V1"
        assert drive_copy.exists(), (
            "Clone failed and no Drive copy found. Either make the GitHub repo "
            f"reachable or upload the repo folder to {drive_copy}"
        )
        import shutil as _sh
        _sh.copytree(drive_copy, REPO_DIR)
        print("Using repo copy from Drive")

try:
    from packaging.requirements import Requirement
except ImportError:
    _pip(["install", "-q", "packaging"])
    from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _meta_version

# Parse requirements-colab.txt; strip any torch* line (Colab rule #1: the
# preinstalled torch/torchvision/torchaudio build must never be touched).
reqs = []
for _line in (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    try:
        _r = Requirement(_line)
    except Exception:
        print("⚠ bỏ qua requirement không parse được:", _line)
        continue
    if _r.name.lower().replace("-", "_").startswith("torch"):
        print("skip (never touch Colab torch):", _line)
        continue
    reqs.append(_r)

def _satisfied(r):
    """Installed + in range — via importlib.metadata, WITHOUT importing it."""
    try:
        v = _meta_version(r.name)
    except PackageNotFoundError:
        return False
    return (not r.specifier) or r.specifier.contains(v, prereleases=True)

missing = [r for r in reqs if FORCE_REINSTALL_DEPS or not _satisfied(r)]
did_install = bool(missing)
if missing:
    print(f"installing {len(missing)} package(s):", ", ".join(r.name for r in missing))
    _pip(["install", "-q", "--prefer-binary", *[str(r) for r in missing]])
else:
    print("dependencies satisfied — no pip install needed")

_pip(["install", "-q", "-e", str(REPO_DIR), "--no-deps"])

# faiss: gpu wheel with cpu fallback (metadata check — no import)
def _installed(*names):
    for n in names:
        try:
            _meta_version(n)
            return n
        except PackageNotFoundError:
            pass
    return None

if _installed("faiss-gpu-cu12", "faiss-gpu", "faiss-cpu", "faiss") is None:
    if _pip(["install", "-q", "faiss-gpu-cu12"]) != 0:
        _pip(["install", "-q", "faiss-cpu"])
    did_install = True

# `pip check` + micro-fixes for known conflicts (max 2 rounds, then warn)
def _pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"],
                       capture_output=True, text=True)
    return r.returncode, ((r.stdout or "") + "\n" + (r.stderr or "")).strip()

if did_install:
    rc, out = _pip_check()
    for _round in (1, 2):
        if rc == 0:
            break
        # pip's two REAL formats (round-3 fix L-R3-8 — the old regex missed the
        # version-conflict wording so that repair branch never ran):
        #   "pkgA 1.0 requires pkgB, which is not installed."
        #   "pkgA 1.0 has requirement pkgB<2,>=1, but you have pkgB 3.0."
        _specs = sorted({
            m.strip()
            for m in re.findall(
                r"(?:requires|has requirement) (.+?), (?:but you have|which is not installed)", out)
            if not m.strip().lower().startswith("torch")
        })
        if not _specs:
            break
        print(f"pip check micro-fix (round {_round}):", ", ".join(_specs))
        _pip(["install", "-q", "--prefer-binary", *_specs])
        rc, out = _pip_check()
    print("pip check: OK" if rc == 0 else f"⚠ pip check còn cảnh báo (không chặn):\n{out}")

# Purge stale sys.modules of upgraded packages BEFORE importing cvp (v12).
# ONLY the packages actually (re)installed THIS run (round-11): purging every
# requirement dropped numpy/pandas from sys.modules while torch still held
# references to the old modules — the "NumPy module was reloaded" warning.
if did_install:
    _ALIAS = {"pillow": "pil", "pyyaml": "yaml", "opencv_python_headless": "cv2",
              "open_clip_torch": "open_clip", "scikit_learn": "sklearn"}
    _roots = {r.name.lower().replace("-", "_") for r in missing} | {"cvp", "faiss"}
    _roots |= {_ALIAS[n] for n in _roots & set(_ALIAS)}
    _purged = [m for m in list(sys.modules)
               if m.split(".", 1)[0].lower().replace("-", "_") in _roots]
    for _m in _purged:
        sys.modules.pop(_m, None)
    if _purged:
        print(f"purged {len(_purged)} stale sys.modules entries")

    # Sanity (round-11): the HF stack must import cleanly in a FRESH
    # interpreter — a broken hub/accelerate pairing must surface HERE with an
    # actionable message, not 5 cells later as a cryptic circular import.
    _rc = _run([sys.executable, "-c", "import transformers, accelerate"])
    if _rc != 0:
        print("⚠ transformers/accelerate KHÔNG import được — thường do phiên cài "
              "này đã hạ cấp huggingface-hub dưới mức accelerate cần. Cách sửa "
              "sạch nhất: Runtime ▸ Disconnect and delete runtime, rồi Run all "
              "lại từ đầu (mọi tiến độ đã nằm trên Drive, không mất gì).")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
import cvp
print("cvp", cvp.__version__, "ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 · Point cvp at the data + GPU setup (TF32 / SDPA / bf16)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, torch

os.environ["CVP_PATHS__DATA_ROOT"]      = str(DATA_DIR)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(ARTIFACTS)
os.environ["CVP_SETTINGS"] = str(REPO_DIR / "configs" / "settings.yaml")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
GPU_NAME, VRAM_GB, USE_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    # TF32 fast paths (new API with old fallback)
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    for fn in ("enable_flash_sdp", "enable_mem_efficient_sdp"):
        if hasattr(torch.backends.cuda, fn):
            getattr(torch.backends.cuda, fn)(True)
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.0f} GB | bf16={USE_BF16}")

# Colab secrets → env (optional: Gemini query enhancement/VQA, HF pushes).
# GOOGLE_API_KEY is the name Colab's built-in "Gemini API key ▸ Import from
# Google AI Studio" button creates — the engine accepts either spelling.
try:
    from google.colab import userdata
    for _sec in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_sec)
            if _v:
                os.environ[_sec] = _v
                print(f"secret {_sec}: loaded")
        except Exception:
            pass
except ImportError:
    pass

from cvp.config import load_settings
from cvp.utils.logging import setup_logging
settings = load_settings()
setup_logging("INFO")
print("data_root      =", settings.paths.data_root)
print("artifacts_root =", settings.paths.artifacts_root)

In [ ]:
# ── 5 · (optional) Auto-extract organiser zips still sitting in data/ ──
# If you uploaded raw zips (Keyframes_L21.zip, ...) instead of extracted
# folders, this unpacks them into the right places, then removes nothing.
import zipfile
from pathlib import Path

ZIP_DEST = {
    "keyframes":       DATA_DIR / "keyframes",
    "videos":          DATA_DIR / "videos",
    "clip-features":   DATA_DIR / "clip-features-32",
    "map-keyframes":   DATA_DIR / "map-keyframes",
    "media-info":      DATA_DIR / "media-info",
    "objects":         DATA_DIR / "objects",
}

def guess_dest(zname: str):
    # Normalise _/space → '-' so 2026 spelling variants (clip_features,
    # Map Keyframes, keyframe singular, video_…) still route correctly.
    # SPECIFIC families are tested FIRST (review R3-C25): a 2026 rename like
    # "keyframe-map-b1.zip" must never fall into the generic keyframes bucket.
    z = zname.lower().replace("_", "-").replace(" ", "-")
    if "map" in z and "keyframe" in z:                        return ZIP_DEST["map-keyframes"]
    if "clip-feature" in z or "features-32" in z:             return ZIP_DEST["clip-features"]
    if "media-info" in z or "metadata" in z:                  return ZIP_DEST["media-info"]
    if "object" in z:                                         return ZIP_DEST["objects"]
    if z.startswith(("keyframes", "keyframe", "key-frames")): return ZIP_DEST["keyframes"]
    if z.startswith(("videos", "video-")):                    return ZIP_DEST["videos"]
    return None

import re as _re
import shutil as _sh
_VID_DIR_RE = _re.compile(r"^[A-Z]\d{2}_V\d{3}$")   # per-video PAYLOAD dirs, never wrappers

_unknown_zips = []
for zp in sorted(DATA_DIR.glob("*.zip")):
    dest = guess_dest(zp.name)
    if dest is None:
        _unknown_zips.append(zp.name); continue
    marker = dest / f".unzipped-{zp.stem}"
    if marker.exists():
        continue
    print("unzipping", zp.name, "→", dest)
    dest.mkdir(parents=True, exist_ok=True)
    tmp_root = dest.parent / "__tmp_unzip"
    if tmp_root.exists():
        _sh.rmtree(tmp_root)
    with zipfile.ZipFile(zp) as z:
        z.extractall(tmp_root)
    # Walk down through GENUINE wrapper folders only: a single child dir that
    # is NOT video-id-shaped. Handles nested shapes like Videos_L28_a/video/*
    # or Keyframes_L26/keyframes/L26_*/ (review R3-C28) while never mistaking
    # a lone per-video payload dir for a wrapper (review R3-C14).
    src = tmp_root
    while True:
        children = list(src.iterdir())
        if len(children) == 1 and children[0].is_dir() and not _VID_DIR_RE.match(children[0].name):
            src = children[0]
            continue
        break
    # MERGE into dest — never overwrite, never drop a second zip's files.
    kept_existing = 0
    for item in src.iterdir():
        target = dest / item.name
        if not target.exists():
            _sh.move(str(item), str(target))
        elif item.is_dir() and target.is_dir():
            for sub in item.iterdir():
                sub_target = target / sub.name
                if not sub_target.exists():
                    _sh.move(str(sub), str(sub_target))
                else:
                    kept_existing += 1
        else:
            kept_existing += 1
    _sh.rmtree(tmp_root, ignore_errors=True)
    if kept_existing:
        print(f"   giữ nguyên {kept_existing} mục đã tồn tại (không ghi đè)")
    marker.touch()
if _unknown_zips:
    print("\n" + "!" * 70)
    print("⚠ CÁC ZIP KHÔNG NHẬN DIỆN ĐƯỢC (KHÔNG được giải nén — kiểm tra tay!):")
    for _n in _unknown_zips:
        print("   •", _n)
    print("  Nếu đây là gói 2026 với tên mới → giải nén thủ công vào đúng thư mục")
    print("  data/{keyframes,map-keyframes,media-info,clip-features-32,objects,videos}")
    print("  (mapping: docs/DATASET_INGESTION.md §2) rồi chạy lại từ ô này.")
    print("!" * 70)
print("zip check done")

In [ ]:
# ── 6 · Materialize data → local disk TỪ ZIP GỐC (nhanh + miễn nhiễm FUSE) ──
# Round-14 (live-run 5): copytree 177k JPG lẻ qua Drive FUSE mất 3h+ rồi làm
# SẬP luôn cả mount ([Errno 107] Transport endpoint is not connected — mọi
# file sau đó đọc ra ENOENT). Chiến lược mới: copy CÁC FILE ZIP về local
# (ít file, to, đọc tuần tự — đúng kiểu I/O FUSE làm tốt) rồi giải nén tại
# chỗ — nhanh hơn nhiều lần, resume theo TỪNG zip, tự remount khi FUSE chết.
# Nội dung chỉ-có-trên-Drive (keyframes K-batch tự cắt, csv tái dựng…) được
# merge bù ở pha 2. Embedding/OCR/caption đọc 177k JPG từ local như cũ.
import re as _re
import shutil, time, zipfile
from pathlib import Path

# _ensure_drive/_drive_alive: bản HARDENED định nghĩa ở Ô 2 (round-19) —
# biết gỡ mount chết (fusermount -uz) + dọn mountpoint bẩn trước khi mount lại.
_ensure_drive()          # verify-R14: gate dưới stat qua FUSE — mount phải sống

def _kf_visible() -> bool:
    """DriveFS trên VM mới có thể thấy data/ nhưng CHƯA thấy subdir keyframes/
    (round-28, live nb03 run 5: gate này rơi nhầm sang nhánh Drive-direct rồi
    chết ở catalog). listdir cha = cú hích ép nạp metadata; zip Keyframes*
    cũng được chấp nhận — materialize vốn bung từ zip, không cần dir Drive."""
    try:
        list(DATA_DIR.iterdir())
        if (DATA_DIR / "keyframes").exists():
            return True
        _zn = [p.name.lower().replace("_", "-").replace(" ", "-")
               for p in DATA_DIR.glob("*.zip")]
        return any(n.startswith(("keyframes", "keyframe", "key-frames")) for n in _zn)
    except OSError:
        return False

_kf_ok = False
if COPY_KEYFRAMES_LOCAL:
    for _w in range(12):                       # tới 2 phút
        if _kf_visible():
            _kf_ok = True
            break
        print(f"⏳ DriveFS chưa thấy keyframes/ hay Keyframes*.zip — đợi ({_w * 10}s) ...")
        time.sleep(10)
    if not _kf_ok:
        print("⚠ 2 phút vẫn không thấy keyframes/ lẫn zip nguồn — rơi về đọc "
              "thẳng Drive (CHẬM; nếu bất thường: Disconnect and delete runtime).")
if COPY_KEYFRAMES_LOCAL and _kf_ok:
    LOCAL_DATA = Path("/content/data")
    LOCAL_DATA.mkdir(exist_ok=True)
    _ZCACHE = Path("/content/__zip_cache")
    _ZCACHE.mkdir(exist_ok=True)
    _VID_DIR_RE = _re.compile(r"^[A-Z]\d{2}_V\d{3}$")

    def _zip_family(zname: str):
        # CÙNG thứ tự ưu tiên với guess_dest ở ô 5 — một zip phải về đúng
        # MỘT family ở cả hai ô. Videos* trả None: video ở lại Drive (symlink).
        z = zname.lower().replace("_", "-").replace(" ", "-")
        if "map" in z and "keyframe" in z:                        return "map-keyframes"
        if "clip-feature" in z or "features-32" in z:             return "clip-features-32"
        if "media-info" in z or "metadata" in z:                  return "media-info"
        if "object" in z:                                         return "objects"
        if z.startswith(("keyframes", "keyframe", "key-frames")): return "keyframes"
        return None

    def _walk_wrapper(root: Path) -> Path:
        # bỏ các folder bọc ngoài thật sự (Keyframes_L26/keyframes/…) nhưng
        # không bao giờ nhầm một payload dir dạng L21_V001 đơn độc là wrapper
        src = root
        while True:
            ch = list(src.iterdir())
            if len(ch) == 1 and ch[0].is_dir() and not _VID_DIR_RE.match(ch[0].name):
                src = ch[0]
                continue
            return src

    def _merge_into(src: Path, dest: Path) -> int:
        """Move src/* vào dest — không ghi đè, đi sâu 1 cấp cho dir trùng."""
        kept = 0
        dest.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            target = dest / item.name
            if not target.exists():
                shutil.move(str(item), str(target))
            elif item.is_dir() and target.is_dir():
                for sub in item.iterdir():
                    st = target / sub.name
                    if not st.exists():
                        shutil.move(str(sub), str(st))
                    else:
                        kept += 1
            else:
                kept += 1
        return kept

    for _sub in ("keyframes", "map-keyframes", "media-info", "objects", "clip-features-32"):
        dst = LOCAL_DATA / _sub
        _stamp = LOCAL_DATA / f".materialized-{_sub}"
        if _stamp.exists():
            # verify-R14: stamp KHÔNG được che zip mới upload giữa session —
            # còn zip matching chưa có marker local thì phải bung bổ sung.
            _ensure_drive()
            _new = [z for z in sorted(DATA_DIR.glob("*.zip"))
                    if _zip_family(z.name) == _sub
                    and not (LOCAL_DATA / f".unzipped-{_sub}-{z.stem}").exists()]
            if not _new:
                print(f"{_sub}: đã materialize trong session này — skip")
                continue
            print(f"{_sub}: {len(_new)} zip mới sau lần materialize trước → bung bổ sung")
        if dst.exists():
            for stale in dst.glob("*.__tmp"):
                shutil.rmtree(stale, ignore_errors=True) if stale.is_dir() else stale.unlink()

        # PHA 1 — bung từ zip nguồn (marker LOCAL theo từng zip → resume mịn;
        # crash giữa merge không sao: lần sau bung lại, merge chỉ bù file thiếu)
        _ensure_drive()
        for zp in sorted(DATA_DIR.glob("*.zip")):
            if _zip_family(zp.name) != _sub:
                continue
            _done = LOCAL_DATA / f".unzipped-{_sub}-{zp.stem}"
            if _done.exists():
                continue
            t0 = time.time()
            lz = _ZCACHE / zp.name
            tmp_root = _ZCACHE / "__tmp_extract"
            for _attempt in (1, 2, 3):
                try:
                    # verify-R14: MỌI syscall chạm FUSE (stat, copyfile) phải
                    # nằm TRONG retry — zip trước mất nhiều phút extract thuần
                    # local, FUSE có thể chết trong cửa sổ đó.
                    _ensure_drive()
                    _free = shutil.disk_usage("/content").free
                    if _free < zp.stat().st_size * 2.2 + 5e9:
                        raise RuntimeError(          # không retry lỗi hết disk
                            f"Disk local sắp đầy ({_free / 1e9:.0f} GB) — không đủ "
                            f"chỗ bung {zp.name}. Runtime ▸ Disconnect and delete "
                            "runtime để lấy máy mới, hoặc đặt "
                            "COPY_KEYFRAMES_LOCAL=False (chậm hơn nhiều).")
                    shutil.copyfile(zp, lz)             # 1 file to, đọc tuần tự
                    if tmp_root.exists():
                        shutil.rmtree(tmp_root)
                    with zipfile.ZipFile(lz) as z:      # CRC check từng member
                        z.extractall(tmp_root)
                    break
                except (OSError, zipfile.BadZipFile) as e:
                    print(f"   ⚠ {zp.name}: {e!r} — thử lại ({_attempt}/3)")
                    if _attempt == 3:
                        raise
                    time.sleep(5)
            kept = _merge_into(_walk_wrapper(tmp_root), dst)
            shutil.rmtree(tmp_root, ignore_errors=True)
            lz.unlink(missing_ok=True)                  # trả disk ngay
            _done.touch()
            print(f"   {zp.name} → local {_sub}/ ({time.time() - t0:.0f}s"
                  + (f", giữ {kept} mục trùng)" if kept else ")"))

        # PHA 2 — merge phần CHỈ có trên Drive (K-batch tự cắt, upload tay…):
        # 1 lần listdir + exists-check local là rẻ; copy lẻ chỉ cho phần thiếu.
        added = 0
        srcD = DATA_DIR / _sub
        # verify-R14: family chỉ-có-folder (không zip nguồn) → pha 1 chưa hề
        # tạo dst; copy2 vào parent chưa tồn tại sẽ FileNotFoundError.
        dst.mkdir(parents=True, exist_ok=True)
        for _attempt in (1, 2, 3):
            try:
                _ensure_drive()                 # srcD.exists cũng chạm FUSE
                if srcD.exists():
                    for item in sorted(srcD.iterdir()):
                        if item.name.startswith(".unzipped-") or item.name.endswith(".__tmp"):
                            continue
                        target = dst / item.name
                        if target.exists():
                            continue
                        tmp_target = dst / (item.name + ".__tmp")
                        if tmp_target.is_dir():
                            shutil.rmtree(tmp_target)
                        elif tmp_target.exists():
                            tmp_target.unlink()
                        (shutil.copytree if item.is_dir() else shutil.copy2)(item, tmp_target)
                        tmp_target.rename(target)
                        added += 1
                break
            except OSError as e:
                print(f"   ⚠ merge Drive-extras {_sub}: {e!r} — thử lại ({_attempt}/3)")
                if _attempt == 3:
                    raise
                time.sleep(5)
        _stamp.touch()
        _n = sum(1 for _ in dst.iterdir())
        print(f"{_sub}: sẵn sàng local ({_n} mục"
              + (f", +{added} bù từ Drive" if added else "") + ")")
    # INTEGRITY + SELF-HEAL (round-11/12, live-run lessons): Google Drive FUSE
    # can serve freshly-written files back EMPTY (buffered writes lost when a
    # session dies mid-sync). Round-11 hit 873 header-less map csvs; round-12
    # hit empty clip-features .npy files that killed the provided_clip32 lane
    # AFTER 8h of GPU work. Validate every LOCAL small-file artifact and heal
    # broken ones straight FROM THE SOURCE ZIP (uploaded long ago = reliably
    # synced), repairing the Drive copy too.
    import zipfile as _zf
    import numpy as _np

    def _bad_csv(f):
        try:
            if f.stat().st_size < 40:
                return True
            with open(f, encoding="utf-8-sig") as fh:
                return sum(1 for _ in fh) < 2          # header only / empty
        except OSError:
            return True

    def _bad_npy(f):
        try:
            if f.stat().st_size < 90:                  # npy header alone is ~64B
                return True
            return _np.load(f, mmap_mode="r").shape[0] == 0
        except Exception:
            return True

    def _bad_empty(f):
        try:
            return f.stat().st_size == 0
        except OSError:
            return True

    # (subdir, glob, zip-name matcher, validator, key depth 1=basename 2=vid/name)
    _HEAL_SPECS = [
        ("map-keyframes", "*.csv",
         lambda z: "map" in z and "keyframe" in z, _bad_csv, 1),
        ("clip-features-32", "*.npy",
         lambda z: "clip-feature" in z or "features-32" in z, _bad_npy, 1),
        ("media-info", "*.json",
         lambda z: "media-info" in z or "metadata" in z, _bad_empty, 1),
        ("objects", "*/*.json",
         lambda z: "object" in z, _bad_empty, 2),
    ]
    for _sub, _pat, _match, _isbad, _depth in _HEAL_SPECS:
        _dirL = LOCAL_DATA / _sub
        if not _dirL.is_dir():
            continue
        _key = (lambda p: p.name) if _depth == 1 else (lambda p: f"{p.parent.name}/{p.name}")
        _bad = [f for f in sorted(_dirL.glob(_pat)) if _isbad(f)]
        if not _bad:
            print(f"{_sub} integrity: OK")
            continue
        print(f"⚠ {len(_bad)} file LOCAL rỗng/hỏng trong {_sub}/ (Drive FUSE mất "
              "dữ liệu?) — tự phục hồi từ zip gốc ...")
        # Zip handles opened ONCE per family (round-13): re-opening a Drive
        # zip per bad file would stall for hours on a family-scale corruption.
        _ensure_drive()                        # verify-R14: ZipFile đọc qua FUSE
        _members, _open_zips = {}, []
        for _z in DATA_DIR.glob("*.zip"):
            _zl = _z.name.lower().replace("_", "-")
            if _match(_zl):
                _zh = _zf.ZipFile(_z)
                _open_zips.append(_zh)
                for _n in _zh.namelist():
                    if not _n.endswith("/"):
                        _parts = Path(_n).parts
                        _members["/".join(_parts[-_depth:])] = (_zh, _n)
        _healed = 0
        for f in _bad:
            _srcz = _members.get(_key(f))
            if not _srcz:
                continue
            _data = _srcz[0].read(_srcz[1])
            if not _data:
                continue
            f.write_bytes(_data)                       # heal LOCAL
            _drv = DATA_DIR / _sub / _key(f)           # heal DRIVE too
            try:
                if not _drv.exists() or _isbad(_drv):
                    _tmpf = _drv.parent / (_drv.name + ".__tmp")
                    _tmpf.write_bytes(_data)
                    _tmpf.replace(_drv)
            except OSError:
                pass
            _healed += 1
        for _zh in _open_zips:
            _zh.close()
        print(f"   phục hồi {_healed}/{len(_bad)}")
        _still = [_key(f) for f in _bad if _isbad(f)]
        if _still:
            raise RuntimeError(
                f"{len(_still)} file trong {_sub}/ vẫn hỏng sau phục hồi "
                f"(vd {_still[:3]}) — kiểm tra zip nguồn còn trong data/ trên "
                "Drive (đừng xóa zip!) rồi chạy lại ô này.")
    # videos stay on Drive (huge); link them in
    _ensure_drive()
    if (DATA_DIR / "videos").exists() and not (LOCAL_DATA / "videos").exists():
        (LOCAL_DATA / "videos").symlink_to(DATA_DIR / "videos")
    import os
    os.environ["CVP_PATHS__DATA_ROOT"] = str(LOCAL_DATA)
    from cvp.config import load_settings
    settings = load_settings()
    print("data_root now:", settings.paths.data_root)
else:
    print("using Drive data_root directly")

In [ ]:
# ── 7 · Catalog + self-extract K-batch keyframes (+ sync back to Drive) ──
import time
from contextlib import contextmanager

@contextmanager
def _log_stage(name: str):
    """Tee nhẹ kiểu v12: ghi start/end/duration của mỗi stage lên Drive."""
    log_path = ARTIFACTS / "logs" / "nb01.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} START {name}\n")
    try:
        yield
    finally:
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} END   {name} "
                    f"({time.time() - t0:.0f}s)\n")

from cvp.data.catalog import KeyframeCatalog
from cvp.data.extraction import extract_missing

# TransNetV2 for K-batch shot detection (best detector per the winning teams).
# --no-deps: torch/numpy/opencv already exist — Colab torch must not be touched.
# ffmpeg-python is REQUIRED by predict_video() (round-3 fix M-R3-1) and itself
# hard-imports `past.builtins` from the `future` distribution at import time
# (round-4 fix: `future` is a REAL runtime dep, not a py2 leftover). All three
# packages DO declare dependencies, but every one of them is either already on
# Colab or installed by this very call — so --no-deps stays torch-safe (round-5
# wording fix C-R5-2; the ffmpeg BINARY ships with Colab).
# Missing/failed install is fine: extraction falls back to PySceneDetect.
if INSTALL_TRANSNETV2:
    def _transnet_ready() -> bool:
        try:
            import transnetv2_pytorch  # noqa: F401
            import ffmpeg  # noqa: F401
            return True
        except ImportError:
            return False

    if _transnet_ready():
        print("TransNetV2 + ffmpeg-python: already installed")
    else:
        _pip(["install", "-q", "--no-deps", "transnetv2-pytorch", "ffmpeg-python", "future"])
        # Verify-then-report: a green pip rc alone proved nothing in round 3.
        print("TransNetV2:", "READY" if _transnet_ready()
              else "unavailable — SceneDetect fallback will be used")

with _log_stage("catalog"):
    n_extracted = extract_missing(settings)     # K-videos without keyframes
    print("extracted videos:", n_extracted)
    catalog = KeyframeCatalog(settings)
    df = catalog.build(force=FORCE_CATALOG or n_extracted > 0)
    # Round-11 (live-run): a manifest built while the map csvs read EMPTY says
    # 0 has_map forever (the corpus signature ignores map csvs) — detect the
    # poisoned state and rebuild once the csvs are healthy again.
    if len(df) and int(df.has_map.sum()) == 0:
        _map_dir = Path(str(settings.paths.data_root)) / "map-keyframes"
        _valid = (sum(1 for f in _map_dir.glob("*.csv") if f.stat().st_size > 40)
                  if _map_dir.is_dir() else 0)
        if _valid:
            print(f"manifest nói 0 map nhưng {_valid} csv hợp lệ đang có "
                  "→ FORCE rebuild catalog")
            df = catalog.build(force=True)
if len(df) and int(df.has_map.sum()) == 0:
    raise RuntimeError(
        "TOÀN BỘ catalog KHÔNG có map-keyframes — frame_idx nộp bài sẽ SAI. "
        "DỪNG tại đây thay vì tốn nhiều giờ embed vô ích. Kiểm tra thông báo "
        "phục hồi map csv ở ô 6, xác nhận data/map-keyframes trên Drive có "
        "csv thật (mở thử 1 file), rồi chạy lại ô này."
    )
print(f"catalog: {len(df):,} keyframes / {df.video_id.nunique()} videos "
      f"({int(df.has_map.sum()):,} frames with map-keyframes)")
_no_map = int((~df.has_map).sum())
if _no_map:
    print(f"⚠ {_no_map:,} keyframes KHÔNG có map-keyframes → frame_idx đang là "
          "ƯỚC LƯỢNG, nộp bài sẽ SAI. Upload gói map-keyframes của BTC, hoặc "
          "tái dựng từ video gốc: python scripts/05_rebuild_map_keyframes.py "
          "(cần videos/*.mp4), rồi chạy lại ô này với FORCE_CATALOG=True.")

# K-batch sync-back: khi COPY_KEYFRAMES_LOCAL=True, extract_missing ghi
# keyframes + map-keyframes mới vào data_root LOCAL (/content/data) — local
# disk BỐC HƠI khi hết session, NB3/laptop sẽ không bao giờ thấy chúng.
# Mirror mọi folder/CSV mà Drive CHƯA có về DATA_DIR (tmp + rename như ô 6).
# No-op khi chạy thẳng trên Drive hoặc không có gì mới.
from pathlib import Path
import shutil as _sh

n_synced = 0
_local_root = Path(str(settings.paths.data_root)).resolve()
if _local_root != DATA_DIR.resolve():
    for src_root, dst_root, want_dir in (
        (_local_root / "keyframes", DATA_DIR / "keyframes", True),
        (_local_root / "map-keyframes", DATA_DIR / "map-keyframes", False),
    ):
        if not src_root.is_dir():
            continue
        for item in sorted(src_root.iterdir()):
            if item.name.endswith(".__tmp"):
                continue
            if not (item.is_dir() if want_dir else item.suffix == ".csv"):
                continue
            target = dst_root / item.name
            if target.exists():
                continue
            dst_root.mkdir(parents=True, exist_ok=True)
            tmp = dst_root / (item.name + ".__tmp")
            if tmp.is_dir():
                _sh.rmtree(tmp)
            elif tmp.exists():
                tmp.unlink()
            (_sh.copytree if want_dir else _sh.copy2)(item, tmp)
            tmp.rename(target)
            n_synced += 1
print(f"sync-back to Drive: {n_synced} item(s)" if n_synced
      else "sync-back: nothing new for Drive")

# Organiser CLIP features → free extra retrieval lane, but ONLY with FULL
# coverage: ingest_provided_features bỏ qua video không có .npy, rồi
# store.build sẽ hard-fail trên lane thiếu vector (video K-batch tự extract
# KHÔNG BAO GIỜ có organiser features).
feat_dir = settings.paths.data(settings.paths.clip_features_dir)
if feat_dir.is_dir() and "provided_clip32" not in EMBED_MODELS:
    _vids = set(map(str, df.video_id.unique()))
    _have = {p.stem for p in feat_dir.glob("*.npy")}
    _missing_feats = sorted(_vids - _have)
    if not _missing_feats:
        EMBED_MODELS = EMBED_MODELS + ["provided_clip32"]
        print("auto-added 'provided_clip32' to EMBED_MODELS (full .npy coverage)")
    else:
        print(f"provided_clip32 lane skipped: {len(_missing_feats)} video(s) have no "
              f"organiser features (K-batch present?) — e.g. {_missing_feats[:3]}")

In [ ]:
# ── 5 (01b) · Captions ONLY — đi NGƯỢC danh sách video ──
# Phiên nb01 chính caption L21→L30; phiên boost này L30→L21. Resume-skip theo
# từng video làm hai phiên hội tụ ở giữa mà không cần điều phối gì thêm.
# LƯU Ý: CAPTION_STRIDE ở ô PARAMS phải GIỐNG HỆT giá trị bên nb01.
from cvp.auxindex.captioner import caption_all_keyframes

vids = [str(v) for v in df.video_id.unique()][::-1]
print(f"captions-boost: {len(vids)} videos, đi ngược từ {vids[0]} về {vids[-1]}, "
      f"stride={CAPTION_STRIDE}")
with _log_stage("captions-boost"):
    n = caption_all_keyframes(settings, catalog, videos=vids, stride=CAPTION_STRIDE)
print(f"✅ captions-boost: {n} video mới trong phiên này")
print("Khi CẢ HAI phiên đều xong captions: chạy lại nb01 một lượt với "
      "FORCE_TEXT_INDEX=True để BM25 nạp đủ trường caption cho toàn bộ 873 video.")